[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/mooseInferenceNotebook.ipynb)

# **MOOSE Inference (Task 1): Download DICOM from IDC, convert to NIfTI, run moosez segmentation**

This notebook is the GPU-side of the MOOSE twoVM workflow. It:
1. Downloads DICOM series from Imaging Data Commons
2. Converts each series to NIfTI via dcm2niix
3. Runs moosez inference with the requested clinical CT models
4. Tars + lz4-compresses the moosez output tree as `moose_segmentations.tar.lz4`

Post-processing (DICOM-SEG generation) happens in Task 2 on a CPU-only VM.

Please cite:

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nat Methods 18, 203-211 (2021). https://doi.org/10.1038/s41592-020-01008-z

Li X, Morgan PS, Ashburner J, Smith J, Rorden C. (2016) The first step for neuroimaging data analysis: DICOM to NIfTI conversion. J Neurosci Methods. 264:47-56.

## Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import yaml
from idc_index.index import IDCClient

# ── wall-clock anchor ──────────────────────────────────────────────────────
NOTEBOOK_START = time.time()

def _elapsed(since=None):
    """Return seconds since `since` (default: NOTEBOOK_START) as a string."""
    return f"{time.time() - (since if since is not None else NOTEBOOK_START):.1f}s"

def _dir_size_mb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024 ** 2)

# ── environment snapshot ───────────────────────────────────────────────────
print(f"Python        : {sys.version}")
print(f"Working dir   : {os.getcwd()}")
print(f"Disk (/)      : ", end="")
_df = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
print(_df.stdout.splitlines()[-1] if _df.returncode == 0 else "unavailable")

try:
    import importlib.metadata as _im
    for _pkg in ("moosez", "nnunetv2", "SimpleITK", "torch", "idc_index", "papermill"):
        try:
            print(f"  {_pkg:<20} {_im.version(_pkg)}")
        except Exception:
            print(f"  {_pkg:<20} (not installed)")
except Exception as _e:
    print(f"Version check failed: {_e}")

print(f"\n[T+{_elapsed()}] Imports complete")


## Parameters

The cell below is tagged `parameters` so papermill can inject values at runtime.
When running interactively, edit the values directly.

In [ ]:
# Papermill injects SeriesInstanceUIDs as a Python list when called with
#   papermill notebook.ipynb out.ipynb -y "SeriesInstanceUIDs: [uid1, uid2]"
SeriesInstanceUIDs = [
    "1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533"
]

# If empty, downloads from IDC using SeriesInstanceUIDs (default).
# If set to a gs:// URI, downloads everything under that path instead
# and sorts by SeriesInstanceUID DICOM tag.
# e.g. "gs://my-bucket/dicom/cohort1"
input_uri = ""

# GCP project where HMAC secrets live. Only needed when input_uri is set.
secret_project = ""

# Comma-separated list of moosez clinical CT model names
# Available: clin_ct_body, clin_ct_body_composition, clin_ct_cardiac,
#            clin_ct_digestive, clin_ct_lungs, clin_ct_muscles,
#            clin_ct_organs, clin_ct_peripheral_bones, clin_ct_ribs,
#            clin_ct_vertebrae
moose_models = "clin_ct_body,clin_ct_body_composition,clin_ct_cardiac,clin_ct_digestive,clin_ct_lungs,clin_ct_muscles,clin_ct_organs,clin_ct_peripheral_bones,clin_ct_ribs,clin_ct_vertebrae"

# 'cuda' for GPU, 'cpu' for CPU-only
accelerator = "cuda"

# Optional GCS prefix (gs://bucket/prefix) for per-series checkpoints so a preempted
# VM resumes finished series instead of recomputing the whole batch. Empty = disabled.
checkpoint_gcs = ""


## Normalize parameters

In [ ]:
import re


def _flatten_uids(raw):
    """Normalize any papermill input shape (str / list / dict) to a clean
    list of SeriesInstanceUIDs.

    Tolerates malformed YAML where UIDs are separated by whitespace rather
    than commas, e.g. `[uid1, uid2 uid3 uid4]` — YAML parses the third item
    as one long string, so we re-split on whitespace+commas and drop
    anything that isn't a UID-shaped token (digits and dots only).
    """
    if isinstance(raw, str):
        try:
            parsed = yaml.safe_load(raw)
        except Exception:
            parsed = raw
        if isinstance(parsed, dict):
            parsed = parsed.get("SeriesInstanceUIDs", list(parsed.values()))
        raw = parsed
    items = list(raw) if isinstance(raw, (list, tuple)) else [raw]

    flat = []
    for item in items:
        if item is None:
            continue
        flat.extend(p for p in re.split(r"[\s,]+", str(item).strip()) if p)

    cleaned = [u for u in flat if re.fullmatch(r"[\d.]+", u)]
    seen, deduped = set(), []
    for u in cleaned:
        if u not in seen:
            seen.add(u)
            deduped.append(u)
    return deduped


if input_uri:
    if not input_uri.startswith("gs://"):
        raise ValueError(f"input_uri must start with gs:// — got {input_uri!r}")
    if not secret_project:
        raise ValueError("secret_project must be set when input_uri is set")
    series_uids = []
else:
    series_uids = _flatten_uids(SeriesInstanceUIDs)
    if not series_uids:
        raise ValueError(
            f"No valid SeriesInstanceUIDs parsed from input: {SeriesInstanceUIDs!r}"
        )

models = [m.strip() for m in moose_models.split(",") if m.strip()]

if input_uri:
    print(f"Input URI         : {input_uri}  (series will be discovered after download)")
else:
    print(f"Series to process : {len(series_uids)}")
    for u in series_uids:
        print(f"  {u}")
print(f"MOOSE models      : {models}")
print(f"Accelerator       : {accelerator}")

In [ ]:
## Checkpoint setup — NIfTI bundle (convert-first) + per-(series, model) resume
import hashlib
import shlex

DICOM_DIR = Path("/tmp/dicom")
NIFTI_DIR = Path("/tmp/nifti")
MOOSE_OUT_DIR = Path("/tmp/moose_output")
for _d in (DICOM_DIR, NIFTI_DIR, MOOSE_OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)


class Checkpointer:
    """Resume a preempted inference VM from GCS instead of recomputing.

    Two layers, both namespaced by run_key = hash(series set + models):
      nifti/all.tar.lz4           one bundle of every converted NIfTI, written once after
                                  the conversion phase (skip re-download + convert on restart)
      seg/<uid>/<model>.tar.lz4   one model's output for one series (skip finished inference)

    No-op when gcs_uri is empty. Auth = VM pet service account (ADC)."""

    def __init__(self, gcs_uri, run_key):
        self.enabled = bool(gcs_uri)
        if not self.enabled:
            print("Checkpointing OFF (checkpoint_gcs not set)")
            return
        if not gcs_uri.startswith("gs://"):
            raise ValueError(f"checkpoint_gcs must start with gs:// — got {gcs_uri!r}")
        try:
            from google.cloud import storage
        except ImportError as exc:
            raise RuntimeError(
                "checkpoint_gcs is set but google-cloud-storage is not installed in "
                "this image — rebuild the inference image (see Dockerfile)."
            ) from exc
        bucket_name, _, prefix = gcs_uri[len("gs://"):].partition("/")
        self._bucket = storage.Client().bucket(bucket_name)
        self._root = f"{prefix.rstrip('/')}/moose_ckpt/{run_key}".lstrip("/")
        print(f"Checkpointing ON -> gs://{bucket_name}/{self._root}")

    # ── low-level GCS + tar helpers ─────────────────────────────────────────
    def _names(self, sub):
        if not self.enabled:
            return set()
        base = f"{self._root}/{sub}/"
        return {b.name[len(base):] for b in self._bucket.list_blobs(prefix=base)
                if b.name != base}

    def _upload(self, sub, name, local):
        self._bucket.blob(f"{self._root}/{sub}/{name}").upload_from_filename(str(local))

    def _download(self, sub, name, local):
        self._bucket.blob(f"{self._root}/{sub}/{name}").download_to_filename(str(local))

    @staticmethod
    def _tar(src_root, members, out_file):
        joined = " ".join(shlex.quote(str(m)) for m in members)
        subprocess.run(
            f"tar -cf - -C {shlex.quote(str(src_root))} {joined} | lz4 > {shlex.quote(str(out_file))}",
            shell=True, check=True)

    @staticmethod
    def _untar(archive, dest_root):
        Path(dest_root).mkdir(parents=True, exist_ok=True)
        subprocess.run(
            f"lz4 -d -c {shlex.quote(str(archive))} | tar -xf - -C {shlex.quote(str(dest_root))}",
            shell=True, check=True)

    # ── NIfTI layer (single bundle, written once after conversion) ──────────
    def restore_nifti_bundle(self, nifti_dir):
        """Extract the NIfTI bundle if present; return the set of uids restored."""
        if not self.enabled or not self._bucket.blob(f"{self._root}/nifti/all.tar.lz4").exists():
            return set()
        local = Path("/tmp/ckpt_nifti_all.tar.lz4")
        self._download("nifti", "all.tar.lz4", local)
        self._untar(local, nifti_dir)
        local.unlink(missing_ok=True)
        return {d.name for d in Path(nifti_dir).iterdir() if d.is_dir()}

    def save_nifti_bundle(self, uids, nifti_dir):
        if not self.enabled or not uids:
            return
        local = Path("/tmp/ckpt_nifti_all.tar.lz4")
        self._tar(nifti_dir, sorted(uids), local)
        self._upload("nifti", "all.tar.lz4", local)
        local.unlink(missing_ok=True)

    # ── segmentation layer (per series + model) ─────────────────────────────
    def seg_done(self):
        done = set()
        for n in self._names('seg'):
            if '/' in n and n.endswith('.tar.lz4'):
                uid, fn = n.split('/', 1)
                done.add((uid, fn[:-len('.tar.lz4')]))
        return done

    def restore_seg(self, uid, model, out_dir):
        local = Path(f"/tmp/ckpt_seg_{uid}_{model}.tar.lz4")
        self._download('seg', f'{uid}/{model}.tar.lz4', local)
        self._untar(local, Path(out_dir) / uid)
        local.unlink(missing_ok=True)

    def save_seg(self, uid, model, members, series_root):
        if not self.enabled or not members:
            return
        local = Path(f"/tmp/ckpt_seg_{uid}_{model}.tar.lz4")
        self._tar(series_root, members, local)
        self._upload('seg', f'{uid}/{model}.tar.lz4', local)
        local.unlink(missing_ok=True)


def do_checkpoint_restore():
    """Compute run_key from the now-final series_uids+models and restore from GCS.

    Called below -- immediately, in IDC mode, where series_uids is already final
    -- or after the GCS download cell in GCS mode, where series_uids isn't final
    until that download+sort completes. Either way this runs exactly once, before
    anything downstream (the IDC download cell's skip-already-restored check, or
    dcm2niix/MOOSE) needs restored_nifti_uids/completed_seg.
    """
    global run_key, checkpointer, restored_nifti_uids, completed_seg
    run_key = hashlib.sha1(
        ("|".join(sorted(series_uids)) + "::" + ",".join(sorted(models))).encode()
    ).hexdigest()[:16]

    checkpointer = Checkpointer(checkpoint_gcs, run_key)
    restored_nifti_uids = checkpointer.restore_nifti_bundle(NIFTI_DIR)
    completed_seg = checkpointer.seg_done()
    for uid, model in sorted(completed_seg):
        checkpointer.restore_seg(uid, model, MOOSE_OUT_DIR)

    print(f"\n[T+{_elapsed()}] Checkpoint setup complete  "
          f"(NIfTI bundle restored: {len(restored_nifti_uids)} series; "
          f"(series,model) done: {len(completed_seg)}; "
          f"{len(series_uids) - len(restored_nifti_uids)} series still need download/convert)")


# IDC mode: series_uids is already final here (from SeriesInstanceUIDs), so restore
# now -- before the IDC download cell below -- so it can skip already-checkpointed
# series instead of re-downloading them. GCS mode restores later, once series_uids
# is known: see the cell after the GCS download below.
if not input_uri:
    do_checkpoint_restore()


## GPU availability check

If `accelerator=cuda` is requested but no GPU is present, fall back to CPU rather than failing.

In [ ]:
usage_metrics = {"gpu": [], "series": {}}

# ── PyTorch / CUDA info ────────────────────────────────────────────────────
_t_torch = time.time()
try:
    import torch
    print(f"PyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version    : {torch.version.cuda}")
        print(f"cuDNN version   : {torch.backends.cudnn.version()}")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            free_b, total_b = torch.cuda.mem_get_info(i)
            print(f"  GPU {i}: {props.name}  |  "
                  f"VRAM total {total_b/1024**3:.1f} GB  free {free_b/1024**3:.1f} GB  |  "
                  f"SM {props.major}.{props.minor}  multiprocessors {props.multi_processor_count}")
    print(f"PyTorch import  : {time.time()-_t_torch:.1f}s")
except Exception as _exc:
    print(f"PyTorch unavailable: {_exc}")

# ── nvidia-smi usage metrics ───────────────────────────────────────────────
try:
    import nvidia_smi
    nvidia_smi.nvmlInit()
    for i in range(nvidia_smi.nvmlDeviceGetCount()):
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
        name = nvidia_smi.nvmlDeviceGetName(handle)
        mem = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        util = nvidia_smi.nvmlDeviceGetUtilizationRates(handle)
        usage_metrics["gpu"].append({
            "index": i,
            "name": name if isinstance(name, str) else name.decode(),
            "total_mb": mem.total // (1024 ** 2),
            "free_mb": mem.free // (1024 ** 2),
            "gpu_util_pct": util.gpu,
        })
    print(f"\nGPU(s) found: {usage_metrics['gpu']}")
except Exception as exc:
    print(f"nvidia-smi unavailable: {exc}")
    if accelerator == "cuda":
        print("WARNING: accelerator=cuda but no GPU found; falling back to cpu")
        accelerator = "cpu"

print(f"\n[T+{_elapsed()}] GPU check complete")


## Download DICOM from IDC

In [ ]:
if not input_uri:
    DICOM_DIR.mkdir(parents=True, exist_ok=True)

    client = IDCClient()
    download_errors = []
    sort_errors = []  # not applicable in IDC mode -- GCS mode populates this instead

    print(f"[T+{_elapsed()}] Starting download of {len(series_uids)} series")

    for uid in series_uids:
        if uid in restored_nifti_uids:
            usage_metrics["series"].setdefault(uid, {})["checkpoint_restored"] = True
            print(f"  Skipping download for {uid} (NIfTI restored from checkpoint)")
            continue

        dest = DICOM_DIR / uid
        dest.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        print(f"  Downloading {uid} ...", flush=True)
        try:
            client.download_from_selection(
                downloadDir=str(dest),
                seriesInstanceUID=uid,
            )
            elapsed = round(time.time() - t0, 1)
            dcm_files = list(dest.rglob("*.dcm"))
            size_mb = _dir_size_mb(dest)
            throughput = size_mb / elapsed if elapsed > 0 else 0
            usage_metrics["series"].setdefault(uid, {})["download_s"] = elapsed
            usage_metrics["series"][uid]["download_dcm_files"] = len(dcm_files)
            usage_metrics["series"][uid]["download_mb"] = round(size_mb, 1)
            print(f"  Downloaded {uid}: {len(dcm_files)} DICOM files  "
                  f"{size_mb:.1f} MB  in {elapsed}s  ({throughput:.1f} MB/s)")

            # ── DICOM image-dimension metadata ────────────────────────────────
            rows, cols = None, None
            try:
                import pydicom
                if dcm_files:
                    _ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
                    rows = int(getattr(_ds, "Rows", 0)) or None
                    cols = int(getattr(_ds, "Columns", 0)) or None
            except Exception:
                pass
            n_slices = len(dcm_files)
            total_pixels = rows * cols * n_slices if (rows and cols) else None
            usage_metrics["series"][uid]["slices"] = n_slices
            usage_metrics["series"][uid]["rows"] = rows
            usage_metrics["series"][uid]["cols"] = cols
            usage_metrics["series"][uid]["total_pixels"] = total_pixels
            _dim = f"{rows}x{cols}" if (rows and cols) else "?x?"
            _pix = f"  |  total pixels {total_pixels:,}" if total_pixels else ""
            print(f"    Dimensions  : {_dim} px  |  {n_slices} slices{_pix}")

        except Exception as exc:
            msg = f"{uid}: {traceback.format_exc()}"
            download_errors.append(msg)
            print(f"  ERROR downloading {uid}: {exc}")

    if download_errors:
        Path("download_error_file.txt").write_text("\n".join(download_errors))

    print(f"\n[T+{_elapsed()}] Download phase complete  "
          f"({len(series_uids)-len(download_errors)-len(restored_nifti_uids)} downloaded, "
          f"{len(restored_nifti_uids)} from checkpoint, {len(download_errors)} failed)")


## Download DICOM from GCS

Used instead of the IDC download above when `input_uri` is set. Downloads everything
under `input_uri` with `s5cmd` (authenticated via GCS HMAC keys pulled from Secret
Manager), then sorts the files into `DICOM_DIR/<uid>/` by reading each file's
SeriesInstanceUID tag.

In [ ]:
if input_uri:
    DICOM_DIR.mkdir(parents=True, exist_ok=True)

    bucket, _, prefix = input_uri[len("gs://"):].partition("/")

    print(f"[T+{_elapsed()}] Fetching HMAC credentials from Secret Manager (project={secret_project})")
    try:
        from google.cloud import secretmanager
    except ImportError:
        print("google-cloud-secret-manager not installed in this image -- installing now "
              "(rebuild the inference image to avoid this at runtime; see Dockerfile)")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "google-cloud-secret-manager"],
            check=True,
        )
        from google.cloud import secretmanager

    _sm_client = secretmanager.SecretManagerServiceClient()

    def _access_secret(secret_id: str) -> str:
        name = f"projects/{secret_project}/secrets/{secret_id}/versions/latest"
        return _sm_client.access_secret_version(name=name).payload.data.decode("utf-8").strip()

    access_key = _access_secret("s5cmd-hmac-key-id")
    secret_key = _access_secret("s5cmd-hmac-secret")
    Path("~/.aws").expanduser().mkdir(parents=True, exist_ok=True)
    Path("~/.aws/credentials").expanduser().write_text(
        f"[default]\naws_access_key_id = {access_key}\naws_secret_access_key = {secret_key}\n"
    )

    staging = Path("/tmp/dicom_staging")
    staging.mkdir(parents=True, exist_ok=True)
    s3_path = f"s3://{bucket}/{prefix}/*" if prefix else f"s3://{bucket}/*"
    print(f"[T+{_elapsed()}] Downloading {s3_path} -> {staging} via s5cmd", flush=True)
    t0 = time.time()
    subprocess.run([
        "s5cmd", "--endpoint-url", "https://storage.googleapis.com",
        "cp", s3_path, str(staging) + "/"
    ], check=True)
    print(f"  Downloaded {_dir_size_mb(staging):.1f} MB in {round(time.time() - t0, 1)}s")

    print(f"[T+{_elapsed()}] Sorting downloaded files by SeriesInstanceUID", flush=True)
    import pydicom

    # No per-series download step exists in bulk/GCS mode (the whole prefix is one
    # all-or-nothing s5cmd call, already raised above on failure via check=True), so
    # download_errors stays empty here -- it keeps its IDC-mode meaning (a series
    # whose download failed, used downstream to exclude it from conversion). Per-file
    # parse/sort failures below are a different kind of problem and go in sort_errors.
    download_errors = []
    sort_errors = []
    for dcm_file in staging.rglob("*.dcm"):
        try:
            ds = pydicom.dcmread(str(dcm_file), stop_before_pixels=True)
            uid = str(ds.SeriesInstanceUID)
            dest = DICOM_DIR / uid
            dest.mkdir(parents=True, exist_ok=True)
            shutil.move(str(dcm_file), str(dest / dcm_file.name))
        except Exception as exc:
            sort_errors.append(f"{dcm_file}: {exc}")
    shutil.rmtree(staging)

    series_uids = sorted({d.name for d in DICOM_DIR.iterdir() if d.is_dir()})
    print(f"  Sorted into {len(series_uids)} series  ({len(sort_errors)} file(s) failed to sort)")

    for uid in series_uids:
        dest = DICOM_DIR / uid
        dcm_files = list(dest.rglob("*.dcm"))
        size_mb = _dir_size_mb(dest)
        usage_metrics["series"].setdefault(uid, {})["download_dcm_files"] = len(dcm_files)
        usage_metrics["series"][uid]["download_mb"] = round(size_mb, 1)

        rows, cols = None, None
        try:
            if dcm_files:
                _ds = pydicom.dcmread(str(dcm_files[0]), stop_before_pixels=True)
                rows = int(getattr(_ds, "Rows", 0)) or None
                cols = int(getattr(_ds, "Columns", 0)) or None
        except Exception:
            pass
        n_slices = len(dcm_files)
        total_pixels = rows * cols * n_slices if (rows and cols) else None
        usage_metrics["series"][uid]["slices"] = n_slices
        usage_metrics["series"][uid]["rows"] = rows
        usage_metrics["series"][uid]["cols"] = cols
        usage_metrics["series"][uid]["total_pixels"] = total_pixels
        _dim = f"{rows}x{cols}" if (rows and cols) else "?x?"
        _pix = f"  |  total pixels {total_pixels:,}" if total_pixels else ""
        print(f"  {uid}: {len(dcm_files)} DICOM files  {size_mb:.1f} MB  |  {_dim} px  |  {n_slices} slices{_pix}")

    if sort_errors:
        Path("download_error_file.txt").write_text("\n".join(sort_errors))

    print(f"\n[T+{_elapsed()}] Download phase complete  "
          f"({len(series_uids)} series discovered, {len(sort_errors)} file(s) failed to sort)")


In [ ]:
# GCS mode: series_uids wasn't final until the download+sort above completed, so
# the checkpoint restore that IDC mode already did before its download happens
# here instead.
if input_uri:
    do_checkpoint_restore()


## Convert DICOM -> NIfTI with dcm2niix

In [ ]:
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

dcm2niix_errors = []
converted_uids = []
failed_download_uids = {e.split(":")[0] for e in download_errors}
_to_convert = [u for u in series_uids
               if u not in failed_download_uids and u not in restored_nifti_uids]

print(f"[T+{_elapsed()}] Starting dcm2niix conversion for {len(_to_convert)} series")

for uid in _to_convert:
    dcm_path = DICOM_DIR / uid
    nii_path = NIFTI_DIR / uid
    nii_path.mkdir(parents=True, exist_ok=True)

    dcm_files = list(dcm_path.rglob("*.dcm"))
    print(f"  Converting {uid}  ({len(dcm_files)} DICOMs) ...", flush=True)
    t0 = time.time()
    result = subprocess.run(
        ["dcm2niix", "-z", "y", "-f", "%i_%s", "-o", str(nii_path), str(dcm_path)],
        capture_output=True,
        text=True,
    )
    elapsed = round(time.time() - t0, 1)

    nii_files = list(nii_path.glob("*.nii.gz"))
    if result.returncode != 0 or not nii_files:
        msg = f"{uid}: return_code={result.returncode}\n{result.stderr}"
        dcm2niix_errors.append(msg)
        print(f"  ERROR converting {uid} (return={result.returncode})")
        if result.stderr:
            print(f"    stderr: {result.stderr[:500]}")
    else:
        nii_sizes = {f.name: round(f.stat().st_size / (1024**2), 1) for f in nii_files}
        usage_metrics["series"].setdefault(uid, {})["dcm2niix_s"] = elapsed
        usage_metrics["series"][uid]["nifti_files_mb"] = nii_sizes
        converted_uids.append(uid)
        print(f"  Converted  {uid}: {elapsed}s  ->  "
              + ", ".join(f"{n} ({s} MB)" for n, s in nii_sizes.items()))

if dcm2niix_errors:
    Path("dcm2niix_error_file.txt").write_text("\n".join(dcm2niix_errors))

# Checkpoint all NIfTIs as a single bundle, once, before any model runs — so a later
# (preempted) attempt restores them and skips re-download + re-conversion. Only written
# when we actually converted something new this attempt (otherwise the bundle already exists).
if converted_uids:
    all_nifti_uids = sorted({d.name for d in NIFTI_DIR.iterdir() if d.is_dir()})
    try:
        checkpointer.save_nifti_bundle(all_nifti_uids, NIFTI_DIR)
        print(f"  [checkpoint] saved NIfTI bundle ({len(all_nifti_uids)} series)")
    except Exception as _ckpt_exc:
        print(f"  WARNING: NIfTI bundle checkpoint failed: {_ckpt_exc}")

print(f"\n[T+{_elapsed()}] dcm2niix complete  "
      f"({len(converted_uids)}/{len(_to_convert)} succeeded)")


## Run MOOSE inference

moosez writes to `<MOOSE_OUT_DIR>/<uid>/moosez-<model>-<timestamp>/segmentations/` for each
requested model. The whole tree is compressed into a single archive below.

In [ ]:

# ── pre-import moosez / SimpleITK so import time is visible separately ────
print(f"[T+{_elapsed()}] Importing moosez and SimpleITK ...", flush=True)
_t_import = time.time()
from moosez import moose
import SimpleITK
from moosez import image_processing as _ip
print(f"[T+{_elapsed()}] moosez + SimpleITK imported in {time.time()-_t_import:.1f}s")


# ── helper: sample GPU memory via nvidia-smi ──────────────────────────────
def _gpu_mem_snapshot(label: str) -> dict:
    snap = {"label": label, "t": round(time.time() - NOTEBOOK_START, 1)}
    try:
        import nvidia_smi
        for i in range(nvidia_smi.nvmlDeviceGetCount()):
            h = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
            m = nvidia_smi.nvmlDeviceGetMemoryInfo(h)
            u = nvidia_smi.nvmlDeviceGetUtilizationRates(h)
            snap[f"gpu{i}_used_mb"] = m.used // (1024**2)
            snap[f"gpu{i}_free_mb"] = m.free // (1024**2)
            snap[f"gpu{i}_util_pct"] = u.gpu
    except Exception:
        pass
    return snap

# ── helper: log a GPU snapshot to stdout and append to metrics list ───────
_gpu_snapshots = []
def _log_gpu(label: str):
    s = _gpu_mem_snapshot(label)
    _gpu_snapshots.append(s)
    parts = [f"T+{s['t']}s  {label}"]
    for k, v in s.items():
        if k.startswith("gpu"):
            parts.append(f"{k}={v}")
    print("  [GPU] " + "  ".join(parts), flush=True)

# ── MOOSE inference loop (series-outer; checkpoints per (series, model)) ───
MOOSE_OUT_DIR.mkdir(parents=True, exist_ok=True)
moose_errors = []

# organ_indices: {model_name: {label_int: organ_name}}
# Captured directly from each Model object's own .organ_indices (moosez's own,
# already-correct parsing of its dataset.json) the first time a model succeeds --
# not re-derived from dataset.json here, so it can't drift from how moosez itself
# reads its own files. SNOMED coding is NOT attached here: the post-process step
# resolves each label name against the curated moose_snomed_mapping.csv, so the
# (static) mapping isn't baked into every inference run's output.
organ_indices = {}

# Process every series that has a NIfTI available (freshly converted or restored).
_ready = set(converted_uids) | restored_nifti_uids
inference_uids = [u for u in series_uids if u in _ready]

print(f"\n[T+{_elapsed()}] Starting MOOSE inference for {len(inference_uids)} series  "
      f"({len(models)} models each); {len(completed_seg)} (series,model) already checkpointed\n")

for uid in inference_uids:
    nii_dir = NIFTI_DIR / uid
    out_path = MOOSE_OUT_DIR / uid
    out_path.mkdir(parents=True, exist_ok=True)
    stats_dir = out_path / "stats"
    stats_dir.mkdir(exist_ok=True)

    nii_files = list(nii_dir.glob("*.nii.gz"))
    if not nii_files:
        msg = f"{uid}: no NIfTI files found in {nii_dir}"
        moose_errors.append(msg)
        print(f"ERROR: {msg}")
        continue
    nii_file = nii_files[0]
    print(f"[T+{_elapsed()}] Series {uid}  "
          f"({round(nii_file.stat().st_size/(1024**2), 1)} MB)")
    _log_gpu("before-series")

    series_start = time.time()
    series_model_times = {}

    for model_idx, model in enumerate(models):
        if (uid, model) in completed_seg:
            print(f"  [{model_idx+1}/{len(models)}] {model}  (restored from checkpoint)")
            continue

        print(f"\n  [{model_idx+1}/{len(models)}] {model}  T+{_elapsed()}", flush=True)
        _log_gpu(f"before-{model}")

        # Snapshot existing top-level entries so we can checkpoint only what this
        # model produces (robust to the moosez output directory layout).
        before = {p.name for p in out_path.iterdir()}

        t0 = time.time()
        try:
            seg_paths, model_objs = moose(str(nii_file), [model], str(out_path), accelerator)
            inference_elapsed = round(time.time() - t0, 1)
            series_model_times[model] = inference_elapsed

            seg_files = list(out_path.rglob("*.nii.gz"))
            seg_sizes = {Path(f).name: round(Path(f).stat().st_size/(1024**2), 1)
                         for f in (seg_paths or [])}
            _log_gpu(f"after-{model}")
            print(f"  inference : {inference_elapsed}s  |  "
                  f"seg files so far: {len(seg_files)}  |  new segs: {seg_sizes}")

            if seg_paths and model_objs and model not in organ_indices:
                organ_indices[model] = dict(model_objs[0].organ_indices)

            # ── stats CSVs ────────────────────────────────────────────────
            stats_files = []
            if seg_paths and model_objs:
                t_stats = time.time()
                try:
                    model_obj = model_objs[0]
                    seg_image = SimpleITK.ReadImage(str(seg_paths[0]))
                    raw_image = SimpleITK.ReadImage(str(nii_file))

                    vol_csv = stats_dir / f"{model_obj.multilabel_prefix}{uid}_volume.csv"
                    _ip.get_shape_statistics(seg_image, model_obj, str(vol_csv))

                    if seg_image.GetSize() != raw_image.GetSize():
                        resampler = SimpleITK.ResampleImageFilter()
                        resampler.SetReferenceImage(raw_image)
                        resampler.SetInterpolator(SimpleITK.sitkNearestNeighbor)
                        seg_image = resampler.Execute(seg_image)
                    int_csv = stats_dir / f"{model_obj.multilabel_prefix}{uid}_CT_intensity.csv"
                    _ip.get_intensity_statistics(raw_image, seg_image, model_obj, str(int_csv))

                    stats_elapsed = round(time.time() - t_stats, 1)
                    series_model_times[f"{model}_stats_s"] = stats_elapsed
                    stats_files = [p for p in (vol_csv, int_csv) if p.exists()]
                    print(f"  stats     : {stats_elapsed}s  |  "
                          f"{', '.join(p.name for p in stats_files)}")
                except Exception as stats_exc:
                    print(f"  WARNING: stats failed for {model}: {stats_exc}")

            # ── checkpoint this (series, model) ───────────────────────────
            new_entries = [p.name for p in out_path.iterdir() if p.name not in before]
            members = new_entries + [f"stats/{p.name}" for p in stats_files]
            if seg_paths and members:
                try:
                    checkpointer.save_seg(uid, model, members, out_path)
                    completed_seg.add((uid, model))
                    print(f"  [checkpoint] saved {uid}/{model}")
                except Exception as _ckpt_exc:
                    print(f"  WARNING: checkpoint upload failed for {uid}/{model}: {_ckpt_exc}")

        except Exception as exc:
            msg = f"{uid}/{model}: {traceback.format_exc()}"
            moose_errors.append(msg)
            print(f"  ERROR {model}: {exc}")

    total_elapsed = round(time.time() - series_start, 1)
    usage_metrics["series"].setdefault(uid, {})["moose_s"] = total_elapsed
    usage_metrics["series"][uid]["moose_models_s"] = series_model_times
    usage_metrics["series"][uid]["gpu_snapshots"] = _gpu_snapshots.copy()
    _gpu_snapshots.clear()

    _log_gpu("after-series")
    print(f"\n[T+{_elapsed()}] Series {uid} done: "
          f"{total_elapsed}s total  ({len(series_model_times)} model(s) run this attempt)")

if moose_errors:
    Path("moose_errors.txt").write_text("\n".join(moose_errors))

print(f"\n[T+{_elapsed()}] MOOSE inference phase complete  "
      f"({len(inference_uids)} series processed, {len(moose_errors)} model error(s))")


## Package outputs

In [ ]:
print(f"[T+{_elapsed()}] Starting output packaging")


def compress_dir(src_dir: Path, out_file: str) -> None:
    """Tar a directory and compress with lz4."""
    t0 = time.time()
    src_raw_mb = _dir_size_mb(src_dir)
    print(f"  Compressing {src_dir.name}  ({src_raw_mb:.1f} MB uncompressed) ...", flush=True)
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    elapsed = round(time.time() - t0, 1)
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    ratio = src_raw_mb / size_mb if size_mb > 0 else 0
    print(f"  Created {out_file}  ({size_mb:.1f} MB compressed  ratio {ratio:.1f}x  in {elapsed}s)")


# Models entirely restored from checkpoint never went through moose() above, so
# organ_indices has no entry for them yet -- back-fill from a Model instance
# directly (cheap: weights are already on disk, no inference/download involved).
_missing_models = [m for m in models if m not in organ_indices]
if _missing_models:
    try:
        from moosez.models import Model as _Model
        from moosez import system as _msystem
        _output_manager = _msystem.OutputManager(False, False)
        for _model_name in _missing_models:
            try:
                _model_obj = _Model(_model_name, _output_manager)
                organ_indices[_model_name] = dict(_model_obj.organ_indices)
            except Exception as _model_exc:
                print(f"  WARNING: could not back-fill organ indices for {_model_name}: {_model_exc}")
    except Exception as _exc:
        print(f"  WARNING: could not back-fill organ indices for checkpoint-restored models: {_exc}")

try:
    (MOOSE_OUT_DIR / "moose_organ_indices.json").write_text(json.dumps(organ_indices, indent=2))
    _total_labels = sum(len(labels) for labels in organ_indices.values())
    print(f"  Bundled organ indices for {len(organ_indices)} model(s): {list(organ_indices)}")
    print(f"  Label names captured: {_total_labels} (SNOMED resolved in post-processing)")
except Exception as _exc:
    print(f"  WARNING: could not bundle organ indices: {_exc}")


compress_dir(MOOSE_OUT_DIR, "moose_segmentations.tar.lz4")
print(f"\n[T+{_elapsed()}] Packaging complete")


## Write usage metrics

In [ ]:
import csv

usage_metrics["total_elapsed_s"] = round(time.time() - NOTEBOOK_START, 1)

# ── Flatten per-(series, model) timing into a CSV for easy analysis ────────
# Column prefixes mark the scope of each value: run_* is one value for the
# whole notebook execution, series_* is per-series (repeated across that
# series' model rows), model_* varies per (series, model) row.
gpu0 = usage_metrics["gpu"][0] if usage_metrics["gpu"] else {}
csv_path = Path("moose_inference_UsageMetrics.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "SeriesInstanceUID", "model",
        "series_checkpoint_restored",
        "series_download_s", "series_download_dcm_files", "series_download_mb",
        "series_rows", "series_cols", "series_slices", "series_total_pixels",
        "series_dcm2niix_s", "series_moose_s",
        "model_inference_s", "model_stats_s",
        "run_gpu_name", "run_gpu_total_mb", "run_total_elapsed_s",
    ])
    for uid, series_metrics in usage_metrics["series"].items():
        model_times = series_metrics.get("moose_models_s", {})
        model_names = [m for m in models if m in model_times] or [""]
        for model in model_names:
            writer.writerow([
                uid,
                model,
                series_metrics.get("checkpoint_restored", False),
                series_metrics.get("download_s", ""),
                series_metrics.get("download_dcm_files", ""),
                series_metrics.get("download_mb", ""),
                series_metrics.get("rows", ""),
                series_metrics.get("cols", ""),
                series_metrics.get("slices", ""),
                series_metrics.get("total_pixels", ""),
                series_metrics.get("dcm2niix_s", ""),
                series_metrics.get("moose_s", ""),
                model_times.get(model, ""),
                model_times.get(f"{model}_stats_s", ""),
                gpu0.get("name", ""),
                gpu0.get("total_mb", ""),
                usage_metrics["total_elapsed_s"],
            ])

# Full usage_metrics (including the per-series gpu_snapshots time series and
# nifti_files_mb breakdown, which don't fit the flat CSV) is printed here for
# debugging via the executed output notebook's cell output -- not written to
# a separate file.
print(json.dumps(usage_metrics, indent=2))
print(f"\n[T+{_elapsed()}] Usage metrics written ({csv_path.name})")


## Summary

In [ ]:
total_s = round(time.time() - NOTEBOOK_START, 1)

print("=" * 70)
print("MOOSE Inference Summary")
print("=" * 70)
print(f"  Input source          : {input_uri if input_uri else 'IDC'}")
print(f"  Total wall-clock time : {total_s}s")
print(f"  Total series          : {len(series_uids)}")
print(f"  Download failures     : {len(download_errors) + len(sort_errors)}")
print(f"  dcm2niix failures     : {len(dcm2niix_errors)}")
print(f"  MOOSE failures        : {len(moose_errors)}")
print(f"  Models used           : {models}")
print()

# ── per-series phase breakdown ─────────────────────────────────────────────
for uid, metrics in usage_metrics.get("series", {}).items():
    print(f"  {uid}")
    dl_s   = metrics.get("download_s", 0)
    dl_mb  = metrics.get("download_mb", "?")
    dcm_n  = metrics.get("download_dcm_files", "?")
    conv_s = metrics.get("dcm2niix_s", 0)
    moose_s = metrics.get("moose_s", 0)

    rows   = metrics.get("rows")
    cols   = metrics.get("cols")
    slices = metrics.get("slices")
    total_pixels = metrics.get("total_pixels")
    _dim   = f"{rows}x{cols}" if (rows and cols) else "?x?"
    _pix   = f"  total pixels {total_pixels:,}" if total_pixels else ""
    _slices = f"{slices}" if slices is not None else "?"
    print(f"    dimensions  : {_dim} px  |  {_slices} slices{_pix}")
    print(f"    download    : {dl_s:>7.1f}s  ({dcm_n} DICOM files  {dl_mb} MB)")
    print(f"    dcm2niix    : {conv_s:>7.1f}s")

    model_times = metrics.get("moose_models_s", {})
    inference_total = 0
    stats_total = 0
    for m in models:
        inf_s   = model_times.get(m, 0)
        stat_s  = model_times.get(f"{m}_stats_s", 0)
        inference_total += inf_s
        stats_total += stat_s
        print(f"    {m:<35}  infer {inf_s:>6.1f}s  stats {stat_s:>5.1f}s")

    print(f"    {'inference subtotal':<35}  {inference_total:>7.1f}s")
    print(f"    {'stats subtotal':<35}  {stats_total:>7.1f}s")
    print(f"    {'moose total (all models)':<35}  {moose_s:>7.1f}s")
    print()

# ── cohort totals ──────────────────────────────────────────────────────────
_all_series = usage_metrics.get("series", {})
_cohort_dcm   = [m.get("download_dcm_files", 0) for m in _all_series.values() if m.get("download_dcm_files") is not None]
_cohort_mb    = [m.get("download_mb", 0)         for m in _all_series.values() if m.get("download_mb") is not None]
_cohort_sl    = [m.get("slices", 0)              for m in _all_series.values() if m.get("slices") is not None]
_cohort_pix   = [m.get("total_pixels", 0)        for m in _all_series.values() if m.get("total_pixels") is not None]

if _all_series:
    print("  Cohort totals:")
    print(f"    Series                : {len(_all_series)}")
    if _cohort_dcm:
        print(f"    DICOM files total     : {sum(_cohort_dcm):,}  "
              f"(min {min(_cohort_dcm):,}  max {max(_cohort_dcm):,}  "
              f"avg {sum(_cohort_dcm)//len(_cohort_dcm):,} per series)")
    if _cohort_mb:
        print(f"    Download size total   : {sum(_cohort_mb):.1f} MB  "
              f"(min {min(_cohort_mb):.1f}  max {max(_cohort_mb):.1f}  "
              f"avg {sum(_cohort_mb)/len(_cohort_mb):.1f} MB per series)")
    if _cohort_sl:
        print(f"    Slices total          : {sum(_cohort_sl):,}  "
              f"(min {min(_cohort_sl):,}  max {max(_cohort_sl):,}  "
              f"avg {sum(_cohort_sl)//len(_cohort_sl):,} per series)")
    if _cohort_pix:
        print(f"    Total pixels (all)    : {sum(_cohort_pix):,}  "
              f"(min {min(_cohort_pix):,}  max {max(_cohort_pix):,}  "
              f"avg {sum(_cohort_pix)//len(_cohort_pix):,} per series)")
    print()

# ── phase share of total ───────────────────────────────────────────────────
if total_s > 0:
    all_dl   = sum(m.get("download_s", 0) for m in usage_metrics.get("series", {}).values())
    all_conv = sum(m.get("dcm2niix_s", 0) for m in usage_metrics.get("series", {}).values())
    all_moose= sum(m.get("moose_s", 0) for m in usage_metrics.get("series", {}).values())
    other    = total_s - all_dl - all_conv - all_moose
    print("  Phase share of total wall-clock:")
    for label, secs in [("download", all_dl), ("dcm2niix", all_conv),
                        ("moose", all_moose), ("overhead/other", other)]:
        pct = 100 * secs / total_s
        bar = "#" * int(pct / 2)
        print(f"    {label:<20} {secs:>7.1f}s  {pct:>5.1f}%  {bar}")

print("=" * 70)

if len(moose_errors) == len(series_uids):
    raise RuntimeError("All series failed MOOSE inference - see moose_errors.txt")
